<a href="https://colab.research.google.com/github/ghadi1927t-cell/Day-vs.-Night-Classification/blob/main/Day_vs_Night_Classification_and_Time_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Cell 1: Extract the uploaded dataset zip file
!unzip -q "Archive (1).zip"
print("Dataset extracted successfully!")

replace test/day/20151102_122322.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Dataset extracted successfully!


In [6]:
# Cell 2: Core Rule-Based Classification Logic
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def analyze_single_image(image_path, show_plot=False):
    """
    Analyzes a single RGB image using pure rule-based logic to classify Day/Night
    and estimate the specific time of day.
    """
    # 1. Read the image
    bgr_img = cv2.imread(image_path)
    if bgr_img is None:
        return None

    # 2. Manual BGR to RGB conversion (Pure matrix slicing)
    rgb_img = bgr_img[:, :, ::-1]
    height, width, _ = rgb_img.shape

    # 3. Calculate overall brightness manually using Luminosity formula
    R = rgb_img[:, :, 0]
    G = rgb_img[:, :, 1]
    B = rgb_img[:, :, 2]
    gray_manual = 0.299 * R + 0.587 * G + 0.114 * B
    avg_brightness = np.mean(gray_manual)

    # 4. Analyze the sky region (Top 40% of the image)
    top_half_end = int(height * 0.4)
    sky_R = R[0:top_half_end, :]
    sky_G = G[0:top_half_end, :]
    sky_B = B[0:top_half_end, :]

    avg_sky_brightness = (np.mean(sky_R) + np.mean(sky_G) + np.mean(sky_B)) / 3

    # Calculate color warmth ratio in the sky (for Evening/Golden Hour detection)
    avg_sky_R = np.mean(sky_R)
    avg_sky_G = np.mean(sky_G)
    avg_sky_B = np.mean(sky_B)
    warmth_ratio = (avg_sky_R + avg_sky_G) / (2.0 * (avg_sky_B + 1e-5))

    # 5. Rule-Based Decision Logic
    if avg_brightness > 75 or avg_sky_brightness > 80:
        day_night_label = "day"

        if warmth_ratio > 1.35 and avg_brightness < 160:
            estimated_time = "Evening"
        elif avg_brightness > 165:
            estimated_time = "Noon"
        else:
            estimated_time = "Morning"
    else:
        day_night_label = "night"

        if avg_sky_brightness > 45 and warmth_ratio > 1.2:
            estimated_time = "Evening (Late Dusk)"
        else:
            estimated_time = "Night"

    # Optional: Display the individual plot if requested
    if show_plot:
        plt.figure(figsize=(6, 4))
        plt.imshow(rgb_img)
        plt.title(f"Label: {day_night_label} | Time: {estimated_time}")
        plt.axis('off')
        plt.show()

    return day_night_label, estimated_time

In [9]:
# Cell 3: Bulk Dataset Reader and Evaluation Logic
def evaluate_dataset_folder(base_folder_path):
    """
    Loops through 'day' and 'night' subfolders inside the given directory,
    runs the rule-based model on every image, and computes overall classification accuracy.
    """
    # Matching the exact subfolder names (case-sensitive adjustment)
    categories = ['day', 'night']
    total_images = 0
    correct_predictions = 0

    print(f"=== Evaluating Dataset Location: {base_folder_path} ===\n")

    for category in categories:
        folder_path = os.path.join(base_folder_path, category)

        if not os.path.exists(folder_path):
            print(f"Directory missing: {folder_path} (Skipping)")
            continue

        # Filter for valid image formats
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"Processing Category [{category}]: Found {len(images)} images.")

        for img_name in images:
            img_path = os.path.join(folder_path, img_name)

            # Call the rule-based logic function from Cell 2
            outputs = analyze_single_image(img_path, show_plot=False)

            if outputs is None:
                continue # Skip corrupt files

            predicted_label, _ = outputs
            total_images += 1

            # Verify correctness based on the actual subfolder name ('day' or 'night')
            if predicted_label == category:
                correct_predictions += 1

    # Calculate and print performance stats
    if total_images > 0:
        accuracy = (correct_predictions / total_images) * 100
        print("\n=================== FINAL PERFORMANCE ===================")
        print(f"Total Evaluated Images : {total_images}")
        print(f"Correct Classifications: {correct_predictions}")
        print(f"Model Accuracy         : {accuracy:.2f}%")
        print("=========================================================\n")
    else:
        print("No images were found or processed.")

In [11]:
# Cell 4: Run Evaluation on Exact Splits from the Image

print("1) EVALUATING TRAINING DATASET:")
train_folder = "training"
evaluate_dataset_folder(train_folder)

print("\n2) EVALUATING TESTING DATASET:")
test_folder = "test"
evaluate_dataset_folder(test_folder)

1) EVALUATING TRAINING DATASET:
=== Evaluating Dataset Location: Archive\ \(1\)/training ===

Directory missing: Archive\ \(1\)/training/Day (Skipping)
Directory missing: Archive\ \(1\)/training/Night (Skipping)
No images were found or processed.

2) EVALUATING TESTING DATASET:
=== Evaluating Dataset Location: Archive\ \(1\)/test ===

Directory missing: Archive\ \(1\)/test/Day (Skipping)
Directory missing: Archive\ \(1\)/test/Night (Skipping)
No images were found or processed.


<>:4: SyntaxWarning: invalid escape sequence '\ '
<>:8: SyntaxWarning: invalid escape sequence '\ '
<>:4: SyntaxWarning: invalid escape sequence '\ '
<>:8: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_14158/952681672.py:4: SyntaxWarning: invalid escape sequence '\ '
  train_folder = "Archive\ \(1\)/training"
/tmp/ipykernel_14158/952681672.py:8: SyntaxWarning: invalid escape sequence '\ '
  test_folder = "Archive\ \(1\)/test"      # replace with your actual path if different


In [ ]:
# Cell 5: Interactive Visual Demo for a Single Sample Image
# You can replace 'image_001.jpg' with any actual image file name inside your test/day folder
sample_image_path = "test/day/20151101_152050.jpg"

if os.path.exists(sample_image_path):
    print("Running interactive demo on sample image:")
    analyze_single_image(sample_image_path, show_plot=True)
else:
    print(f"Demo note: Update 'sample_image_path' with a real file name from your folder to see the image display.")